# rca-sim — what happened

A live root-cause-analysis loop: Online Boutique runs under steady load, Chaos Mesh breaks one service
at a time, a Datadog monitor notices, and **PRISM** ranks which service is to blame. Every injection is
logged, so every ranking can be marked right or wrong.

This notebook answers four questions, in order:

1. **What ran?** — which faults, on which services, and did the system stay healthy between them.
2. **Did Datadog notice?** — how long detection took, and what it missed.
3. **Did PRISM find it?** — AC@k against ground truth, per fault type.
4. **What did the live setting cost?** — the gap between the estimated anomaly time and the real one,
   which is the thing an offline benchmark can never measure.

Every cell degrades gracefully: if a campaign has not run yet, it says so instead of failing.

> Run with `pip install -e "rca-service[dev]"` from the repo root, then open this file.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "results" else Path.cwd()
sys.path.insert(0, str(ROOT / "rca-service"))

from app.evaluation import load_ground_truth, match, score, unmatched  # noqa: E402

RESULTS = ROOT / "results"

# --- palette -------------------------------------------------------------------------------------
# Categorical slots are assigned in fixed order and never cycled; text uses ink tokens, never a
# series colour, so identity is never carried by colour alone.
SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#87867f", "#e4e3df"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
GOOD, BAD = "#1baf7a", "#e34948"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.linewidth": 1.0, "axes.labelcolor": INK_2,
    "axes.titlesize": 13, "axes.titleweight": "semibold", "axes.titlecolor": INK,
    "axes.titlelocation": "left", "axes.titlepad": 14,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 10, "ytick.labelsize": 10,
    "text.color": INK, "font.size": 11, "legend.frameon": False, "legend.fontsize": 10,
    "grid.color": GRID, "grid.linewidth": 0.8, "lines.linewidth": 2.0, "figure.dpi": 120,
})

def note(message: str) -> None:
    """A visible, non-fatal 'nothing here yet' so the notebook always runs top to bottom."""
    print(f"\u2014 {message}")

def load_incidents() -> list[dict]:
    out = []
    for path in sorted((RESULTS / "incidents").glob("*/report.json")):
        try:
            out.append(json.loads(path.read_text()))
        except json.JSONDecodeError:
            continue
    return out

faults = load_ground_truth(RESULTS / "ground_truth.jsonl")
incidents = load_incidents()
cases = match(faults, incidents) if faults else []
false_alarms = unmatched(incidents, cases) if incidents else []

print(f"{len(faults)} injected faults, {len(incidents)} incidents, {len(false_alarms)} unmatched")

## 1. What ran

The campaign injects one fault at a time and waits out a cooldown longer than the baseline window, so
no incident's reference period contains the previous fault. The table below is the ground truth: it is
written at injection time by `chaos/inject.py` and nothing in the scoring path can see it.

In [ ]:
if not faults:
    note("No faults injected yet. Run `make eval`, or `make overnight` for the whole chain.")
else:
    inventory = pd.DataFrame([{
        "fault": f.fault_type, "target": f.target_service,
        "duration_s": f.t_end - f.t_start,
        "started": pd.to_datetime(f.t_start, unit="s").strftime("%H:%M:%S"),
    } for f in faults])
    display(inventory.groupby(["fault", "target"]).size().unstack(fill_value=0))
    print(f"\n{len(faults)} injections over "
          f"{(max(f.t_end for f in faults) - min(f.t_start for f in faults)) / 3600:.1f} hours")

## 2. Did Datadog notice?

**Time to detect** is the monitor's own latency: from the moment the fault starts to the moment Datadog
fires. It is not something we choose — a 1-minute threshold monitor cannot fire before the condition has
held for a full minute, plus ingestion lag. A fault with no bar was never detected at all, and that is
counted separately rather than hidden.

In [ ]:
if not cases:
    note("No cases yet — nothing to time.")
else:
    # Built from every fault type, not only the detected ones: a type that was never detected must
    # still appear, or the chart quietly omits the very cases the thresholds got wrong.
    kinds = sorted({c.fault.fault_type for c in cases})
    by_type = {f: [c.time_to_detect for c in cases
                   if c.fault.fault_type == f and c.detected and c.time_to_detect is not None]
               for f in kinds}
    missed = {f: sum(1 for c in cases if c.fault.fault_type == f and not c.detected) for f in kinds}
    order = sorted(kinds, key=lambda f: np.median(by_type[f]) if by_type[f] else float("inf"))

    fig, ax = plt.subplots(figsize=(7.5, 0.62 * len(order) + 1.9))
    ax.grid(axis="x", zorder=0)
    ax.set_axisbelow(True)
    for i, fault in enumerate(order):
        values = by_type[fault]
        if not values:
            ax.text(0, i, "  never detected", va="center", ha="left", color=BAD,
                    fontsize=10, zorder=5)
            continue
        # The bar is the median; the individual runs ride on top, so one slow detection cannot
        # hide behind an average.
        ax.barh(i, np.median(values), height=0.55, color=SERIES[0], zorder=3)
        ax.scatter(values, [i] * len(values), s=26, color=SURFACE, edgecolor=INK_2,
                   linewidth=1.2, zorder=4)
        # Past the rightmost dot, not at the median: at the median the text sits under a marker.
        ax.text(max(max(values), np.median(values)), i, f"   {np.median(values):.0f}s median",
                va="center", ha="left", color=INK_2, fontsize=10, zorder=5)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([f"{f}  ({missed[f]} missed)" if missed[f] else f for f in order], color=INK)
    ax.set_xlabel("seconds from injection to alert")
    ax.set_title("Time to detect, by fault type\nbar = median, dots = individual injections")
    ax.set_xlim(0, max([max(v) for v in by_type.values() if v] + [1]) * 1.42)
    plt.tight_layout(); plt.show()

    display(pd.DataFrame([{
        "fault": f,
        "median_s": float(np.median(by_type[f])) if by_type[f] else float("nan"),
        "slowest_s": max(by_type[f]) if by_type[f] else float("nan"),
        "detected": len(by_type[f]), "missed": missed[f],
    } for f in order]).set_index("fault"))

## 3. Did PRISM find it?

**AC@k** is the share of faults where the true service appears in PRISM's top *k*. AC@1 is the strict
reading — the very first guess was right. **Avg@5** averages AC@1..AC@5 into one number, as RCAEval
reports it, so these bars sit directly beside the offline benchmark figures.

The per-fault-type breakdown matters more than the overall number: resource faults and network faults
present completely differently, and one average hides that.

In [ ]:
if not cases:
    note("No scored cases yet.")
else:
    result = score(cases, false_alarms)
    overall, per_type = result["overall"], result["per_fault_type"]
    ks = [1, 3, 5]
    groups = ["overall"] + sorted(per_type)
    values = {k: [overall[f"AC@{k}"]] + [per_type[f][f"AC@{k}"] for f in sorted(per_type)] for k in ks}

    x = np.arange(len(groups))
    width = 0.26
    fig, ax = plt.subplots(figsize=(1.5 * len(groups) + 3, 4.2))
    ax.grid(axis="y", zorder=0)
    ax.set_axisbelow(True)
    for i, k in enumerate(ks):
        # A 2px surface gap between adjacent bars keeps the groups legible.
        ax.bar(x + (i - 1) * width, values[k], width * 0.92, label=f"AC@{k}",
               color=SERIES[i], zorder=3)
        for xi, v in zip(x + (i - 1) * width, values[k]):
            ax.text(xi, v + 0.02, f"{v:.2f}", ha="center", va="bottom", color=INK_2, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(groups, color=INK)
    ax.set_ylim(0, 1.16)
    ax.set_ylabel("share in top k")
    ax.set_title("PRISM accuracy against ground truth")
    # Above the axes, not inside them: at AC@k = 1.00 the bars reach the top and an inside legend
    # sits straight on top of the value labels.
    ax.legend(ncols=3, loc="lower right", bbox_to_anchor=(1, 1.0))
    plt.tight_layout(); plt.show()

    display(pd.DataFrame(per_type).T[["n_faults", "AC@1", "AC@3", "AC@5", "Avg@5",
                                      "missed_detections"]])
    print(f"\nOverall Avg@5 {overall['Avg@5']:.3f} over {overall['n_faults']} faults; "
          f"{overall['missed_detections']} missed, {overall['false_alarms']} false alarms.")

## 4. Anatomy of one incident

This is the case the whole method exists for. A fault in one service makes **its callers** slow, so the
service that *alerts* is usually not the service that *broke*.

Everything is plotted as **deviation from its own baseline, in standard deviations** — which is exactly
the quantity PRISM scores, and it lets metrics of completely different units share one axis honestly.
The root cause moves on an *internal* property (cpu, memory) **and** an external one; a downstream
victim moves only externally, however dramatically.

In [ ]:
def anatomy(case) -> None:
    incident = next((i for i in incidents if i["incident_id"] == case.incident_id), None)
    frame_path = RESULTS / "incidents" / case.incident_id / "metrics.parquet"
    if incident is None or not frame_path.exists():
        note("Stored frame missing for this incident.")
        return
    frame = pd.read_parquet(frame_path)
    t0 = incident["timeline"]["t_anomaly"]
    pre = frame[frame["time"] < t0]

    # The three most deviating columns, which is what PRISM itself looked at. Three series keeps the
    # palette inside its all-pairs limit, and each line is directly labelled.
    scored = {}
    for col in frame.columns:
        if col == "time" or pre[col].std() in (0, np.nan) or not np.isfinite(pre[col].std()):
            continue
        z = (frame[col] - pre[col].mean()) / pre[col].std()
        if np.isfinite(z).any():
            scored[col] = z
    top = sorted(scored, key=lambda c: -np.nanmax(np.abs(scored[c])))[:3]
    if not top:
        note("No column had a usable baseline in this window.")
        return

    minutes = (frame["time"] - t0) / 60
    fig, ax = plt.subplots(figsize=(9, 4.6))
    ax.grid(axis="y", zorder=0)
    ax.set_axisbelow(True)
    ax.axhline(0, color=GRID, linewidth=1)
    for i, col in enumerate(top):
        ax.plot(minutes, scored[col], color=SERIES[i], zorder=3, label=col)

    # Room for the direct labels on the right only -- padding both sides leaves dead space.
    span_x = minutes.max() - minutes.min()
    ax.set_xlim(minutes.min(), minutes.max() + span_x * 0.45)

    # A y in axes fraction via get_xaxis_transform: reading get_ylim() before the lines were drawn
    # pinned these captions near the floor of the finished chart.
    marks = [(0.0, "anomaly", "--"), ((incident["timeline"]["t_trigger"] - t0) / 60,
                                      "monitor fired", ":")]
    for x, label, style in marks:
        if x < minutes.min() or x > minutes.max():
            continue
        ax.axvline(x, color=MUTED, linewidth=1.2, linestyle=style, zorder=2)
        ax.text(x, 0.99, f" {label}", color=MUTED, fontsize=9, va="top",
                transform=ax.get_xaxis_transform())

    # Nudge colliding end labels apart: two series often finish at almost the same deviation, and
    # unmoved the names overprint each other into an unreadable smear.
    low, high = ax.get_ylim()
    gap = (high - low) * 0.075
    placed: list[float] = []
    for y, name in sorted((scored[c].iloc[-1], c) for c in top):
        y = max(y, placed[-1] + gap) if placed else y
        placed.append(y)
        ax.text(minutes.max(), y, f"  {name}", color=INK_2, fontsize=9, va="center")

    ax.set_xlabel("minutes from the estimated anomaly time")
    ax.set_ylabel("deviation from baseline (sigma)")
    ax.set_title(f"{case.fault.fault_type} injected into {case.fault.target_service}\n"
                 f"PRISM ranked: {', '.join(case.ranking[:3])}")
    plt.tight_layout(); plt.show()

    hit = case.fault.target_service in case.ranking[:1]
    print(("Correct: " if hit else "Ranked below the top: ")
          + f"the true root cause was {case.fault.target_service}.")

examples = [c for c in cases if c.detected and c.ranking]
if not examples:
    note("No incident with a ranking yet.")
else:
    # Prefer a propagating fault, where the alerting service and the root cause differ.
    pick = next((c for c in examples if c.fault.fault_type in ("delay", "loss")), examples[0])
    anatomy(pick)

## 5. Why PRISM ranked it that way

PRISM returns an order, not a score. These bars are the evidence behind that order, recomputed from the
same window: for each suspect, how far its strongest metric moved from its own baseline.

Read it as the method does — a service with **both** an internal and an external property deviating is a
cause; one with only external deviation is a symptom.

In [ ]:
if not examples:
    note("No incident to explain yet.")
else:
    incident = next(i for i in incidents if i["incident_id"] == pick.incident_id)
    rows = []
    for suspect in incident["top_suspects"][:5]:
        for metric in suspect["metrics"][:2]:
            sigma = metric["deviation_sigma"]
            if sigma is not None and np.isfinite(sigma):
                rows.append((suspect["position"], suspect["service"],
                             metric["metric"].split("_", 1)[1], sigma))
    if not rows:
        note("No finite deviations recorded for this incident.")
    else:
        rows = sorted(rows, key=lambda r: r[3])[-10:]
        labels = [f"{svc} · {prop}" for _, svc, prop, _ in rows]
        sigmas = [r[3] for r in rows]
        is_target = [r[1] == pick.fault.target_service for r in rows]

        fig, ax = plt.subplots(figsize=(8, 0.42 * len(rows) + 1.8))
        ax.grid(axis="x", zorder=0)
        ax.set_axisbelow(True)
        # Two states, not two categories: the injected service versus everything else. Status colour
        # is paired with the "(injected)" label so it never carries meaning alone.
        ax.barh(range(len(rows)), sigmas, height=0.6, zorder=3,
                color=[GOOD if t else SERIES[0] for t in is_target])
        for i, (value, target) in enumerate(zip(sigmas, is_target)):
            ax.text(value, i, f"  {value:.0f}" + ("  (injected)" if target else ""),
                    va="center", ha="left", color=INK_2, fontsize=9, zorder=4)
        ax.set_yticks(range(len(rows)))
        ax.set_yticklabels(labels, color=INK, fontsize=9)
        ax.set_xlabel("peak deviation from baseline (sigma)")
        ax.set_title("What moved, and by how much\nthe evidence behind PRISM's ranking")
        ax.margins(x=0.18)
        plt.tight_layout(); plt.show()

## 6. What the live setting costs

The offline benchmark hands PRISM the exact injection time. Here nobody knows it: the anomaly time is
*estimated* backwards from when the monitor fired, minus its evaluation window and the ingestion lag.

The gap below is that estimate's error. It is the number this whole setup exists to produce — a
benchmark cannot generate it, because in a benchmark the answer is given.

In [ ]:
gaps = [(i["timeline"]["t_anomaly"] - c.fault.t_start)
        for c in cases if c.detected
        for i in incidents if i["incident_id"] == c.incident_id]
if not gaps:
    note("No detected incidents yet.")
else:
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    ax.grid(axis="y", zorder=0)
    ax.set_axisbelow(True)
    ax.hist(gaps, bins=min(12, max(3, len(gaps))), color=SERIES[0], zorder=3)
    ax.axvline(0, color=MUTED, linewidth=1.2, linestyle="--", zorder=4)
    ax.text(0, ax.get_ylim()[1], " true injection time", color=MUTED, fontsize=9, va="top")
    ax.set_xlabel("estimated anomaly time minus true injection time (seconds)")
    ax.set_ylabel("incidents")
    # Counts are whole numbers; the default locator offers half an incident.
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_title("How far off the estimated anomaly time was\nnegative = estimated too early")
    plt.tight_layout(); plt.show()

    print(f"median {np.median(gaps):+.0f}s, range {min(gaps):+.0f}s to {max(gaps):+.0f}s")
    print("\nTo separate 'PRISM was wrong' from 'the clock was wrong', re-rank the same stored\n"
          "windows using the ground-truth time:  make replay ID=<incident> --at <t_start>")

## 7. How to read all this, and what it does not say

- **A missed detection is not a PRISM failure.** If no monitor fired, PRISM was never asked. Those cases
  are counted separately in section 2 and excluded from AC@k, and a fault type with many misses is
  telling you about thresholds, not about the method.
- **A fault that moves no metric is an unfair miss.** `disk` and `code` in particular need per-service
  configuration to have any effect. Check section 4's plot for the fault type before trusting its score.
- **False alarms include recovery churn.** An alert firing shortly after a fault *ends* is currently
  counted as a false alarm if it falls outside the matching slack. Look at the timestamps before reading
  the false-alarm count as noise.
- **Every incident kept its raw window** in `results/incidents/<id>/metrics.parquet`. Any of this can be
  recomputed later with a different PRISM variant, without re-running the cluster.